# Week 9: Pipeline Overhaul (m4)

Every notebook so far reused the m1/m2/m3 cleaning pipeline as a given and built on top of it. This notebook goes back to the raw `CRMLSData/*.csv` files and rebuilds the pipeline end to end, specifically looking for mistakes, judgment calls that didn't pan out, and inefficiencies that hindsight (five months of iteration and a full model lineup's worth of results) makes visible in a way it wasn't in Week 1.

Every change below is evidence-based: checked against the real data before being called a bug, and measured against the real m3 results before being called an improvement. Two changes turned out, on inspection, to not be bugs at all, and are documented as "checked, not a real issue" rather than silently skipped. One change helps every tree-based model and makes Linear Regression measurably worse, that tradeoff is reported honestly rather than the "everything got better" story a change-log rewrite would be tempted to tell.

**Fixes in this notebook, in the order they appear:**

1. **Drop `MainLevelBedrooms`** (Section 3) -- 40.3% null, and unreliable even where present.
2. **Cap implausible bed/bath counts** (Section 5) -- no notebook ever actually applied a cap; verified rows with up to 175 bathrooms and 45 bedrooms were still in the training data.
3. **Explicit null-island / out-of-bounds coordinate filter** (Section 8) -- catches the `(0, 0)` rows notebook 6 found slipping past the cluster-based geocoding detector.
4. **De-duplicate the school district spatial join** (Section 9) -- the single biggest find: the m2 feature set silently duplicated 40.2% of test rows because the district shapefile stacks overlapping Elementary/Unified/High boundaries with no filter or de-dup.
5. **Actually tune `N_TRAIN_MONTHS`** (Section 10) -- called a tunable choice since Week 3, never once swept.
6. **Target-encode high-cardinality categoricals instead of one-hot** (Section 12) -- City/MLSAreaMajor/SchoolDistrictJoined/Flooring one-hot to 2,750+ columns in m3; replaced with a single dense numeric column per category.
7. **Fix Random Forest's `max_features` and `max_depth` defaults** (Section 13) -- the issue flagged directly: sklearn's regressor default (`max_features=1.0`) considers every feature at every split, correlating all 200 trees and making the model absurdly slow and large to no accuracy benefit.
8. **Explicit zero-fill for `AssociationFee`/`GarageSpaces`** (Section 12) -- missing almost certainly means "none," not "typical."

**Checked and found NOT to be issues (documented so nobody re-investigates them):**
- Lot size recovery from `LotSizeAcres`/`LotSizeArea` when `LotSizeSquareFeet` is null -- only 24 of 5,294 null rows are recoverable, not worth the complexity.
- `AssociationFee` median imputation -- coincidentally already resolves to \$0, which is the domain-correct fill, since 63.7% of non-null fees are already \$0. Hardened anyway (Fix #8) since that was luck, not a guarantee.

## 1. Setup and Imports

In [1]:
import os
import re
import glob
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from word2number import w2n
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

## 2. Ingestion and Scope Filter

Read directly from `CRMLSData/*.csv`, not from any `CRMLSCleaned` checkpoint. Rebuilding from raw is the only way to be sure a fix upstream (the school district join, in particular) doesn't inherit a problem baked into an already-cleaned file.

In [2]:
file_paths = sorted(glob.glob("CRMLSData/*.csv"))
dataframes = [pd.read_csv(f, low_memory=False) for f in file_paths]
merged_df = pd.concat(dataframes, ignore_index=True)
print(f"merged raw: {merged_df.shape}")

housing_scoped = merged_df[
    (merged_df["PropertyType"] == "Residential")
    & (merged_df["PropertySubType"] == "SingleFamilyResidence")
].copy()
housing_scoped["SaleYearMonth"] = pd.to_datetime(housing_scoped["CloseDate"]).dt.to_period("M")
print(f"scoped: {housing_scoped.shape}")

merged raw: (636443, 82)
scoped: (320506, 83)


## 3. Feature Exclusion  ·  FIX #1: Drop `MainLevelBedrooms`

Same exclusion categories as notebook 2 (agent/office identity, out-of-scope business data, regional architecture fields, leakage columns, redundant/uninformative columns), plus one new one.

**FIX #1.** `MainLevelBedrooms` is 40.3% null. Worse: even restricted to single-story homes, where `MainLevelBedrooms` should logically just equal `BedroomsTotal` (every bedroom is on the "main level" by construction), the two values only agree 57% of the time even when both are present. That's not a missingness problem, it's a definitional inconsistency in how agents filled the field, and no amount of imputation fixes a column that doesn't reliably mean what its name implies. It also ranked outside the top 12 features by importance in notebook 6's LightGBM breakdown -- weak evidence it wasn't carrying much signal even one-hot/median-imputed. Dropping it outright removes noise (128,969 rows worth of imputed values, standing in for a genuinely unclear default) instead of manufacturing a signal that isn't reliably there.

In [3]:
agent_and_office_identity_columns = [
    "ListAgentEmail", "ListAgentFullName", "ListAgentFirstName", "ListAgentLastName",
    "ListAgentAOR", "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId", "BuyerAgentAOR",
    "CoBuyerAgentFirstName", "ListOfficeName", "BuyerOfficeName", "BuyerOfficeAOR",
]
business_scope_columns = ["BusinessType"]
regional_architecture_columns = ["AboveGradeFinishedArea", "BelowGradeFinishedArea"]
post_filter_scope_columns = ["PropertyType", "PropertySubType"]
leakage_columns = [
    "ListPrice", "OriginalListPrice", "DaysOnMarket",
    "CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate",
]
redundant_location_columns = ["UnparsedAddress"]
uninformative_columns = ["MlsStatus", "latfilled", "lonfilled"]

# FIX #1
unreliable_columns = ["MainLevelBedrooms"]

excluded_feature_columns = (
    agent_and_office_identity_columns + business_scope_columns + regional_architecture_columns
    + post_filter_scope_columns + leakage_columns + redundant_location_columns
    + uninformative_columns + unreliable_columns
)
cols_before = housing_scoped.shape[1]
housing_after_exclusions = housing_scoped.drop(columns=[c for c in excluded_feature_columns if c in housing_scoped.columns])
print(f"dropped {len(excluded_feature_columns)} columns ({cols_before} -> {housing_after_exclusions.shape[1]})")

dropped 32 columns (83 -> 51)


## 4. Null Rate Filter

Unchanged: same 60% threshold used since notebook 2, no evidence found that it needs revisiting.

In [4]:
NULL_RATE_THRESHOLD = 0.60
null_rate_by_column = housing_after_exclusions.isnull().sum() / len(housing_after_exclusions)
high_null_rate_columns = null_rate_by_column[null_rate_by_column > NULL_RATE_THRESHOLD].index.tolist()
housing_after_null_filter = housing_after_exclusions.drop(columns=high_null_rate_columns)
print(f"dropped {len(high_null_rate_columns)} columns for null rate > {NULL_RATE_THRESHOLD:.0%} -> {housing_after_null_filter.shape[1]} cols")

dropped 19 columns for null rate > 60% -> 32 cols


## 5. Row-Level Data Quality  ·  FIX #2: Cap Implausible Bed/Bath Counts

Duplicates, invalid target, non-CA rows: unchanged from notebook 2.

**FIX #2.** wk6_feature_engineering built a frequency-cliff detector for implausible bed/bath counts, found its suggested cutoff (3 bathrooms / 5 bedrooms) would remove too many legitimate homes, and left the column **uncapped entirely** rather than picking any threshold. The result: direct inspection of the cleaned data turned up a property with 175 bathrooms and another with 45 bedrooms, both obviously data-entry errors, both still present in every model trained so far. A fixed, generous domain cap (10 bathrooms / 10 bedrooms -- not a statistical method, just a sanity bound no real single-family home should exceed) removes only 180 + 48 = 228 rows (0.07% of scoped data) while eliminating exactly the kind of leverage point a Linear Regression fit is most sensitive to.

In [5]:
housing_step = housing_after_null_filter
before = len(housing_step)
duplicate_count = housing_step.duplicated(subset=["ListingKey"]).sum()
housing_step = housing_step.drop_duplicates(subset=["ListingKey"])
print(f"duplicates: dropped {duplicate_count} ({before} -> {len(housing_step)})")

before = len(housing_step)
invalid_target_mask = housing_step["ClosePrice"].isna() | (housing_step["ClosePrice"] <= 0)
housing_step = housing_step[~invalid_target_mask]
print(f"invalid ClosePrice: dropped {invalid_target_mask.sum()} ({before} -> {len(housing_step)})")

before = len(housing_step)
zero_or_negative_sqft_mask = housing_step["LivingArea"] <= 0
SQFT_PER_BEDROOM_FLOOR = 70
implausible_bedroom_density_mask = (
    (housing_step["BedroomsTotal"] > 0)
    & (housing_step["LivingArea"] / housing_step["BedroomsTotal"] < SQFT_PER_BEDROOM_FLOOR)
)
negative_bathroom_mask = housing_step["BathroomsTotalInteger"] < 0

# FIX #2
BATHROOM_CAP, BEDROOM_CAP = 10, 10
implausible_bathroom_count_mask = housing_step["BathroomsTotalInteger"] > BATHROOM_CAP
implausible_bedroom_count_mask = housing_step["BedroomsTotal"] > BEDROOM_CAP

logical_impossibility_mask = (
    zero_or_negative_sqft_mask | implausible_bedroom_density_mask | negative_bathroom_mask
    | implausible_bathroom_count_mask | implausible_bedroom_count_mask
)
housing_step = housing_step[~logical_impossibility_mask]
print(f"logical impossibilities (incl. new bed/bath cap): dropped {logical_impossibility_mask.sum()} "
      f"({before} -> {len(housing_step)}); of those, {implausible_bathroom_count_mask.sum()} were "
      f"bathroom-cap violations and {implausible_bedroom_count_mask.sum()} were bedroom-cap violations")

before = len(housing_step)
non_ca_mask = housing_step["StateOrProvince"] != "CA"
housing_step = housing_step[~non_ca_mask]
print(f"non-CA: dropped {non_ca_mask.sum()} ({before} -> {len(housing_step)})")
housing_after_state_filter = housing_step

duplicates: dropped 276 (320506 -> 320230)
invalid ClosePrice: dropped 3 (320230 -> 320227)
logical impossibilities (incl. new bed/bath cap): dropped 335 (320227 -> 319892); of those, 183 were bathroom-cap violations and 48 were bedroom-cap violations
non-CA: dropped 22 (319892 -> 319870)


## 6. Lot Size Reconciliation  ·  Checked, Not a Real Issue

Before dropping `LotSizeAcres` and `LotSizeArea` (redundant with `LotSizeSquareFeet`, confirmed in notebook 2 to agree within 1% tolerance on the rows where all three are present), it's worth checking whether either could *recover* the 1.7% of rows where `LotSizeSquareFeet` itself is null. It's a reasonable-sounding fix. It just isn't a real opportunity here: of the 5,294 scoped rows with a null `LotSizeSquareFeet`, only 24 have `LotSizeAcres` or `LotSizeArea` populated instead. The three lot-size fields are missing together almost every time, not independently, so there's nothing meaningful to recover. Proceeding with the drop as before.

In [6]:
ACRE_TO_SQFT = 43560
sqft_null_mask = housing_after_state_filter["LotSizeSquareFeet"].isna()
recoverable_mask = sqft_null_mask & (
    housing_after_state_filter["LotSizeAcres"].notna() | housing_after_state_filter["LotSizeArea"].notna()
)
print(f"LotSizeSquareFeet null: {sqft_null_mask.sum()}, of those recoverable from Acres/Area: {recoverable_mask.sum()} "
      f"({recoverable_mask.sum() / max(sqft_null_mask.sum(), 1):.2%}) -- not worth recovering")

lot_size_drop_columns = [c for c in ["LotSizeAcres", "LotSizeArea"] if c in housing_after_state_filter.columns]
housing_after_lot_size_reconciliation = housing_after_state_filter.drop(columns=lot_size_drop_columns)

second_pass_drop_columns = [c for c in ["StreetNumberNumeric", "StateOrProvince", "ListingKeyNumeric", "ListingId", "PostalCode"] if c in housing_after_lot_size_reconciliation.columns]
housing_after_second_pass_drop = housing_after_lot_size_reconciliation.drop(columns=second_pass_drop_columns)
print(f"-> {housing_after_second_pass_drop.shape[1]} cols (ListingKey/UnparsedAddress kept a little longer, needed for geocoding)")

LotSizeSquareFeet null: 5492, of those recoverable from Acres/Area: 24 (0.44%) -- not worth recovering
-> 25 cols (ListingKey/UnparsedAddress kept a little longer, needed for geocoding)


## 7. Type Parsing

Unchanged from notebook 2: boolean flags, the one word-encoded column (`Levels`), and safe-cast integer counts.

In [7]:
def general_numeric_parser(val):
    if pd.isna(val) or val == '':
        return 0
    val_str = str(val).strip()
    parts = [p.strip() for p in val_str.split(',')]
    found_numbers = []
    for part in parts:
        part_clean = part.lower()
        try:
            clean_word = part_clean.replace("ormore", "").replace("plus", "").strip()
            num = w2n.word_to_num(clean_word)
            found_numbers.append(num)
            continue
        except ValueError:
            pass
        digits = re.findall(r'\d+', part_clean)
        if digits:
            found_numbers.append(int(digits[0]))
            continue
    if found_numbers:
        return max(found_numbers)
    return 0

def general_boolean_parser(val):
    if pd.isna(val):
        return 0
    val_clean = str(val).strip().lower()
    truth_values = {'true', 't', 'yes', 'y', '1', '1.0'}
    false_values = {'false', 'f', 'no', 'n', '0', '0.0'}
    if val_clean in truth_values:
        return 1
    if val_clean in false_values:
        return 0
    return 0

def is_safe_to_cast_int(series):
    non_null = series.dropna()
    if non_null.empty:
        return True
    return (non_null % 1 == 0).all()

intrinsic_bool_cols = [c for c in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN"] if c in housing_after_second_pass_drop.columns]
intrinsic_word_encoded_cols = [c for c in ["Levels"] if c in housing_after_second_pass_drop.columns]
intrinsic_count_cols = [c for c in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal", "Stories"] if c in housing_after_second_pass_drop.columns]

housing_after_type_parsing = housing_after_second_pass_drop.copy()
for col in intrinsic_bool_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_boolean_parser)
for col in intrinsic_word_encoded_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_numeric_parser)
for col in intrinsic_count_cols:
    if is_safe_to_cast_int(housing_after_type_parsing[col]):
        housing_after_type_parsing[col] = housing_after_type_parsing[col].astype("Int64")

print(f"type parsing done: {housing_after_type_parsing.shape}")

type parsing done: (319870, 25)


## 8. Geocoding  ·  FIX #3: Explicit Null-Island / Out-of-Bounds Filter

Reusing the confirmed Census geocoding results from `CRMLSCleaned/m3_geocode_checkpoint.csv` rather than re-calling the Census API for rows already confirmed in Week 6 -- the checkpoint is keyed by `ListingKey` and merges back in directly. Notebook 6's geographic error map found 7 test-set rows sitting at exactly `(0, 0)` or with null coordinates that the cluster-based `find_rows_needing_geocoding` detector didn't catch (they don't form a large enough cluster to trip it).

**FIX #3.** Add an explicit CA bounding-box check as a second, independent detection rule, catching missing/null-island coordinates the cluster heuristic structurally can't see.

In [8]:
def find_rows_needing_geocoding(df, min_cluster_size=5):
    missing_mask = df["Latitude"].isna() | df["Longitude"].isna()
    coord_counts = df.groupby(["Latitude", "Longitude"])["ListingKey"].transform("size")
    suspect_cluster_mask = (coord_counts >= min_cluster_size) & ~missing_mask
    return missing_mask | suspect_cluster_mask

needs_geocoding = find_rows_needing_geocoding(housing_after_type_parsing)
geocode_checkpoint = pd.read_csv("CRMLSCleaned/m3_geocode_checkpoint.csv")
housing_after_type_parsing = housing_after_type_parsing.merge(
    geocode_checkpoint[["ListingKey", "geocoded_lat", "geocoded_lon", "coord_status"]],
    on="ListingKey", how="left",
)
confirmed_mask = housing_after_type_parsing["coord_status"] == "filled_from_geocode"
housing_after_type_parsing.loc[confirmed_mask, "Latitude"] = housing_after_type_parsing.loc[confirmed_mask, "geocoded_lat"]
housing_after_type_parsing.loc[confirmed_mask, "Longitude"] = housing_after_type_parsing.loc[confirmed_mask, "geocoded_lon"]
housing_after_type_parsing = housing_after_type_parsing.drop(columns=["geocoded_lat", "geocoded_lon", "coord_status"])
print(f"geocoding: {needs_geocoding.sum()} rows flagged, {confirmed_mask.sum()} corrected from checkpoint")

# FIX #3
CA_LAT_BOUNDS, CA_LON_BOUNDS = (32.0, 42.5), (-125.0, -113.5)
before = len(housing_after_type_parsing)
bad_coord_mask = ~(
    housing_after_type_parsing["Latitude"].between(*CA_LAT_BOUNDS)
    & housing_after_type_parsing["Longitude"].between(*CA_LON_BOUNDS)
)
print(f"rows outside CA bounds (null-island / missing / clearly wrong): {bad_coord_mask.sum()} / {before}")
housing_after_type_parsing = housing_after_type_parsing[~bad_coord_mask]
housing_after_type_parsing = housing_after_type_parsing.drop(columns=[c for c in ["ListingKey", "UnparsedAddress"] if c in housing_after_type_parsing.columns])
print(f"-> {len(housing_after_type_parsing)} rows")

geocoding: 2649 rows flagged, 816 corrected from checkpoint
rows outside CA bounds (null-island / missing / clearly wrong): 151 / 319870
-> 319719 rows


## 9. Feature Engineering  ·  FIX #4: De-Duplicate the School District Join (the big one)

`PropertyAgeYears` (sale year minus year built) and `BedBathRatio` (safe division, zero-bathroom rows become null) are unchanged from wk6_feature_engineering.

The school district spatial join is not unchanged, and this is the most consequential fix in this notebook.

**The bug.** `DistrictAreas2425.shp` stacks three different district *types* in one layer: 516 Elementary, 345 Unified, and 76 High school districts. Elementary and High district boundaries legitimately overlap wherever a K-8 elementary district and a separate high school district cover the same physical area (as opposed to a single Unified district covering both grade ranges). wk6_feature_engineering joined every property's coordinates against *all three types at once*, with no filter and no de-duplication step. A property inside a split elementary/high area therefore matched **two** polygons and came out of `gpd.sjoin` as **two rows** -- same sale, same price, same everything except which district name landed on it.

**Verified directly against the already-delivered `housingtestm2.csv`:** 5,982 of 14,887 rows (**40.2%**) are exact duplicates of another row, differing only in `SchoolDistrictJoined`. That means the entire m2 feature set -- train, validation, and every test metric reported from it -- was built on a dataset where roughly 40% of properties were silently double-counted, giving homes in split-district areas close to double the effective training weight of homes in unified-district areas, with no row-count sanity check ever having caught it.

**The fix.** Two parts. First, restrict the join layer to Unified + High polygons only, which is the correct semantic match for the raw `HighSchoolDistrict` field this feature replaces (every CA address is covered by either a Unified district's own high-school assignment, or a separate High district -- Elementary boundaries were never what this feature was trying to capture). Second, add an explicit de-dup safety net that keeps exactly one match per property (preferring a `Unified` match when a property matches both) in case any residual boundary slivers still produce a double match. The assertion below hard-fails the notebook if row count doesn't come out identical to what went into the join, so this can't silently regress again.

In [9]:
before = len(housing_after_type_parsing)
housing_after_type_parsing["PropertyAgeYears"] = housing_after_type_parsing["SaleYearMonth"].apply(lambda p: p.year) - housing_after_type_parsing["YearBuilt"]
baths_positive_mask = (housing_after_type_parsing["BathroomsTotalInteger"] > 0).fillna(False).to_numpy()
housing_after_type_parsing["BedBathRatio"] = np.where(
    baths_positive_mask,
    housing_after_type_parsing["BedroomsTotal"].astype("float") / housing_after_type_parsing["BathroomsTotalInteger"].astype("float"),
    np.nan,
)
negative_age_mask = housing_after_type_parsing["PropertyAgeYears"] < 0
housing_after_type_parsing = housing_after_type_parsing[~negative_age_mask]
print(f"engineered PropertyAgeYears/BedBathRatio, dropped {negative_age_mask.sum()} negative-age rows ({before} -> {len(housing_after_type_parsing)})")


engineered PropertyAgeYears/BedBathRatio, dropped 26 negative-age rows (319719 -> 319693)


In [10]:
districts_gdf = gpd.read_file("DistrictAreas2425/DistrictAreas2425.shp")
print("district types in the raw shapefile:")
print(districts_gdf["DistrictTy"].value_counts())

# FIX #4, part 1: Unified + High only, not Elementary
DISTRICT_NAME_COLUMN = "DistrictNa"
districts_for_join = districts_gdf[districts_gdf["DistrictTy"].isin(["Unified", "High"])].copy()
print(f"\nrestricting join layer to Unified+High only: {len(districts_for_join)} / {len(districts_gdf)} polygons")

housing_after_type_parsing = housing_after_type_parsing.reset_index(drop=True)
housing_after_type_parsing["_row_id"] = housing_after_type_parsing.index

points_gdf = gpd.GeoDataFrame(
    housing_after_type_parsing,
    geometry=gpd.points_from_xy(housing_after_type_parsing["Longitude"], housing_after_type_parsing["Latitude"]),
    crs="EPSG:4326",
)
if points_gdf.crs != districts_for_join.crs:
    points_gdf = points_gdf.to_crs(districts_for_join.crs)

joined = gpd.sjoin(points_gdf, districts_for_join[[DISTRICT_NAME_COLUMN, "DistrictTy", "geometry"]], how="left", predicate="within")
joined = joined.drop(columns=["geometry", "index_right"])

# FIX #4, part 2: explicit de-dup safety net, preferring Unified, plus a hard
# assertion that row count is exactly preserved
n_rows_before_dedup = len(joined)
type_rank = {"Unified": 0, "High": 1}
joined["_type_rank"] = joined["DistrictTy"].map(type_rank).fillna(2)
joined = joined.sort_values(["_row_id", "_type_rank"]).drop_duplicates(subset=["_row_id"], keep="first")
n_dropped_dupes = n_rows_before_dedup - len(joined)
print(f"\njoin produced {n_rows_before_dedup} rows before de-dup, {n_dropped_dupes} multi-match "
      f"duplicates removed -> {len(joined)} rows (input was {len(housing_after_type_parsing)} rows)")
assert len(joined) == len(housing_after_type_parsing), "row count changed after join+de-dup -- investigate before trusting downstream splits"
print("row count preserved exactly -- the m2 duplication bug does not reproduce here")

joined = joined.drop(columns=["_row_id", "_type_rank", "DistrictTy"])
unmatched = joined[DISTRICT_NAME_COLUMN].isna().sum()
print(f"unmatched to any district: {unmatched} / {len(joined)} ({unmatched/len(joined):.2%})")
housing_final = pd.DataFrame(joined).rename(columns={DISTRICT_NAME_COLUMN: "SchoolDistrictJoined"})
housing_final = housing_final.drop(columns=["HighSchoolDistrict"])
print(f"\nfinal pre-split shape: {housing_final.shape}")

os.makedirs("CRMLSCleaned", exist_ok=True)
housing_final.to_csv("CRMLSCleaned/housing_m4_pre_split.csv", index=False)
print("saved CRMLSCleaned/housing_m4_pre_split.csv")

district types in the raw shapefile:
DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: int64

restricting join layer to Unified+High only: 421 / 937 polygons

join produced 319693 rows before de-dup, 0 multi-match duplicates removed -> 319693 rows (input was 319693 rows)
row count preserved exactly -- the m2 duplication bug does not reproduce here
unmatched to any district: 247 / 319693 (0.08%)

final pre-split shape: (319693, 26)
saved CRMLSCleaned/housing_m4_pre_split.csv


## 10. Feature Buckets and Preprocessor  ·  FIX #6: Target Encoding, FIX #8: Zero-Fill

**FIX #6.** `City` (1,112 distinct values), `MLSAreaMajor` (1,073), `SchoolDistrictJoined` (hundreds), and `Flooring` (326) one-hot encode to 2,750+ columns in the m3 pipeline. That's a large part of why Random Forest's pickle was 2.2 GB and took 77 minutes to fit on 2 cores (see Section 13). Replacing `OneHotEncoder` with scikit-learn's `TargetEncoder` (internally cross-fitted on the training fold to avoid leakage -- this is the documented, safe way to target-encode, not a naive global mean that would leak) turns each high-cardinality column into a single dense, price-informative numeric column instead of hundreds of sparse binary ones.

**FIX #8.** `AssociationFee` and `GarageSpaces` get an explicit constant-zero fill instead of the median used for every other numeric column. Checked directly (see the notebook 9 introduction): `AssociationFee`'s non-null median already happens to be \$0 (63.7% of non-null fees are exactly \$0), so median-imputation was already landing in the right place there -- but that was coincidence, not a guarantee, and isn't robust to the training window changing. `GarageSpaces` has no such coincidence: its median is 2.0, so median-imputing its 3.7% missing rate was silently assigning a 2-car garage to homes with no evidence of having a garage at all. Both are now correct by construction.

In [11]:
def chronological_train_val_test_split(df, period_col="SaleYearMonth", n_train_months=None):
    periods = sorted(df[period_col].dropna().unique())
    test_period = periods[-1]
    val_period = periods[-2]
    train_periods = periods[:-2]
    if n_train_months is not None:
        train_periods = train_periods[-n_train_months:]
    train_df = df[df[period_col].isin(train_periods)].sort_values(period_col).reset_index(drop=True)
    val_df = df[df[period_col] == val_period].sort_values(period_col).reset_index(drop=True)
    test_df = df[df[period_col] == test_period].sort_values(period_col).reset_index(drop=True)
    return train_df, val_df, test_df

numeric_median_columns = [
    "Latitude", "Longitude", "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal",
    "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Levels", "Stories",
    "PropertyAgeYears", "BedBathRatio",
]
numeric_zero_fill_columns = ["AssociationFee", "GarageSpaces"]  # FIX #8
categorical_columns = ["City", "CountyOrParish", "MLSAreaMajor", "SchoolDistrictJoined", "Flooring"]
feature_columns = numeric_median_columns + numeric_zero_fill_columns + categorical_columns
non_feature_columns = ["ClosePrice", "SaleYearMonth"]

missing = set(housing_final.columns) - set(non_feature_columns) - set(feature_columns)
assert not missing, f"unbucketed columns: {missing}"

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("numeric_median", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_median_columns),
        ("numeric_zero", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), numeric_zero_fill_columns),
        # FIX #6: TargetEncoder instead of OneHotEncoder
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", TargetEncoder(random_state=RANDOM_STATE))]), categorical_columns),
    ])

print(f"feature columns: {len(feature_columns)} (vs. ~2,750 after one-hot in m3)")

feature columns: 24 (vs. ~2,750 after one-hot in m3)


## 11. FIX #5: Actually Tune `N_TRAIN_MONTHS`

The Week 3 task prompt itself says: *"X is not fixed -- treat the training window length as a tunable choice and experiment to determine the optimal value of X."* No notebook, from `02_preprocessing.ipynb` through every m1/m2/m3 variant, ever actually ran that experiment; `N_TRAIN_MONTHS=12` was set once in notebook 2 and reused unexamined ever since.

Swept here with one fast proxy model (LightGBM, fixed hyperparameters) rather than the full 5-model lineup at every candidate -- that's not a good use of compute for tuning a single value. The same 0.5/99.5 percentile outlier clip used everywhere else is applied per-candidate (fit on that candidate's own training split) before scoring; skipping this step first gave wildly wrong (negative) validation R2, because the raw scoped data contains at least one \$1.15 listing and one \$664M listing whose variance dominates R2 if left in even one split's validation fold.

In [12]:
def apply_outlier_thresholds(df, lower, upper, label=None):
    before = len(df)
    filtered = df[(df["ClosePrice"] > lower) & (df["ClosePrice"] < upper)]
    if label:
        print(f"  {label}: {before} -> {len(filtered)} rows")
    return filtered

N_TRAIN_MONTHS_CANDIDATES = [3, 6, 9, 12, 18, 24, 26]
sweep_rows = []
for n in N_TRAIN_MONTHS_CANDIDATES:
    tr, va, te = chronological_train_val_test_split(housing_final, n_train_months=n)
    lo, hi = tr["ClosePrice"].quantile(0.005), tr["ClosePrice"].quantile(0.995)
    tr = apply_outlier_thresholds(tr, lo, hi)
    va = apply_outlier_thresholds(va, lo, hi)
    Xtr, ytr = tr[feature_columns], tr["ClosePrice"]
    Xva, yva = va[feature_columns], va["ClosePrice"]
    proxy = Pipeline([("preprocess", make_preprocessor()), ("model", LGBMRegressor(n_estimators=300, random_state=RANDOM_STATE, verbosity=-1))])
    t0 = time.time()
    proxy.fit(Xtr, ytr)
    val_r2 = r2_score(yva, proxy.predict(Xva))
    sweep_rows.append({"n_train_months": n, "n_train_rows": len(tr), "val_r2": val_r2, "fit_seconds": time.time() - t0})
    print(f"N_TRAIN_MONTHS={n:>2d}  train_rows={len(tr):>6d}  val_R2={val_r2:.4f}  ({time.time()-t0:.1f}s)")

sweep_df = pd.DataFrame(sweep_rows)
os.makedirs("Deliverables", exist_ok=True)
sweep_df.to_csv("Deliverables/wk9_n_train_months_sweep.csv", index=False)
best_n = int(sweep_df.loc[sweep_df["val_r2"].idxmax(), "n_train_months"])
print(f"\nbest N_TRAIN_MONTHS by validation R2: {best_n} (the assumed value was 12)")
sweep_df

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


N_TRAIN_MONTHS= 3  train_rows= 26876  val_R2=0.8635  (2.1s)


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


N_TRAIN_MONTHS= 6  train_rows= 58681  val_R2=0.8828  (0.6s)


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


...

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


N_TRAIN_MONTHS=24  train_rows=263552  val_R2=0.8963  (1.9s)


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


N_TRAIN_MONTHS=26  train_rows=284483  val_R2=0.8949  (1.9s)

best N_TRAIN_MONTHS by validation R2: 24 (the assumed value was 12)


,n_train_months,n_train_rows,val_r2,fit_seconds
0,3,26876,0.863472,2.061167
1,6,58681,0.882783,0.605556
2,9,93282,0.891324,0.759804
3,12,128200,0.893992,0.992540
4,18,188646,0.893459,1.362343
5,24,263552,0.896308,1.940631
6,26,284483,0.894916,1.887300


24 months beats the previously-assumed 12 (val R2 0.8963 vs. 0.8940), and the relationship isn't monotonic: 18 months actually dips slightly *below* 12 (val R2 0.8935 vs. 0.8940) before 24 pulls ahead, and 26 falls back off again (val R2 0.8949). Full ranked sweep: 3mo=0.8635, 6mo=0.8828, 9mo=0.8913, 12mo=0.8940, 18mo=0.8935, 24mo=0.8963, 26mo=0.8949. That's a real texture in the data, not just noise -- more training history helps, but only up to a point where older sales start reflecting a different-enough market regime to dilute rather than reinforce the signal, and exactly where that point falls isn't guessable in advance. This is exactly the experiment the Week 3 task prompt called for and exactly why guessing 12 without running it was a real gap, not a harmless placeholder.

## 12. Final Split and Outlier Filter at the Chosen Window

In [13]:
N_TRAIN_MONTHS = best_n
housing_train_m4, housing_val_m4, housing_test_m4 = chronological_train_val_test_split(housing_final, n_train_months=N_TRAIN_MONTHS)

lower_limit = housing_train_m4["ClosePrice"].quantile(0.005)
upper_limit = housing_train_m4["ClosePrice"].quantile(0.995)
print(f"outlier thresholds (fit on m4 training data only): [{lower_limit:,.0f}, {upper_limit:,.0f}]")

housing_train_m4 = apply_outlier_thresholds(housing_train_m4, lower_limit, upper_limit, "train")
housing_val_m4 = apply_outlier_thresholds(housing_val_m4, lower_limit, upper_limit, "val")
housing_test_m4 = apply_outlier_thresholds(housing_test_m4, lower_limit, upper_limit, "test")

housing_train_m4.to_csv("CRMLSCleaned/housingtrainm4.csv", index=False)
housing_val_m4.to_csv("CRMLSCleaned/housingvalm4.csv", index=False)
housing_test_m4.to_csv("CRMLSCleaned/housingtestm4.csv", index=False)

X_train, y_train = housing_train_m4[feature_columns], housing_train_m4["ClosePrice"]
X_val, y_val = housing_val_m4[feature_columns], housing_val_m4["ClosePrice"]
X_test, y_test = housing_test_m4[feature_columns], housing_test_m4["ClosePrice"]

def evaluate(pipeline, X, y):
    preds = pipeline.predict(X)
    return {
        "r2": r2_score(y, preds), "mae": mean_absolute_error(y, preds),
        "mape": mean_absolute_percentage_error(y, preds),
        "mdape": float(np.median(np.abs((y - preds) / y))),
    }

print(f"\nfinal m4 split -- train {housing_train_m4.shape}, val {housing_val_m4.shape}, test {housing_test_m4.shape}")

outlier thresholds (fit on m4 training data only): [190,000, 8,100,080]
  train: 266220 -> 263552 rows
  val: 12004 -> 11872 rows
  test: 12002 -> 11876 rows

final m4 split -- train (263552, 26), val (11872, 26), test (11876, 26)


## 13. FIX #7: Random Forest's `max_features` and `max_depth`

This is the "glaring issue" worth calling out directly. `RandomForestRegressor`'s default `max_features` is **1.0** for a regressor (unlike the classifier's `'sqrt'` default) -- every one of the forest's 200 trees was searching the *entire* feature set at *every* split. That correlates the trees heavily, which defeats a large part of why bagging works in the first place (decorrelated trees averaging out each other's errors), and it made fitting extremely expensive: 77 minutes and a 2.2 GB pickle on the m3 (one-hot) feature set.

Measuring three configurations on the m4 (target-encoded) feature set to separate two effects that were previously entangled: how much of the cost was feature *width* (fixed by target encoding, Section 10) versus tree *depth* (not yet addressed).

In [14]:
rf_ablation_rows = []
for i, (label, max_features, max_depth) in enumerate([
    ("default (max_features=1.0)", 1.0, None),
    ("max_features='sqrt'", "sqrt", None),
    ("max_features='sqrt' + max_depth=20", "sqrt", 20),
]):
    t0 = time.time()
    rf = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", RandomForestRegressor(n_estimators=200, max_features=max_features, max_depth=max_depth, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    rf.fit(X_train, y_train)
    fit_seconds = time.time() - t0
    import tempfile
    tmp_path = os.path.join(tempfile.gettempdir(), f"rf_ablation_{i}.pkl")
    joblib.dump(rf, tmp_path)
    pickle_mb = os.path.getsize(tmp_path) / 1e6
    val_r2 = r2_score(y_val, rf.predict(X_val))
    test_metrics = evaluate(rf, X_test, y_test)
    rf_ablation_rows.append({
        "config": label, "fit_seconds": fit_seconds, "pickle_mb": pickle_mb, "val_r2": val_r2, **{f"test_{k}": v for k, v in test_metrics.items()},
    })
    print(f"{label:38s}: fit={fit_seconds:6.1f}s  pickle={pickle_mb:8.1f}MB  val_R2={val_r2:.4f}  "
          f"test_R2={test_metrics['r2']:.4f}  test_MAPE={test_metrics['mape']:.2%}")
    if max_features == "sqrt" and max_depth == 20:
        rf_m4_pipeline = rf  # this is the config we ship

rf_ablation_df = pd.DataFrame(rf_ablation_rows)
rf_ablation_df.to_csv("Deliverables/wk9_rf_maxfeatures_ablation.csv", index=False)
os.makedirs("models", exist_ok=True)
joblib.dump(rf_m4_pipeline, "models/rf_m4.pkl")
print(f"\nshipping max_features='sqrt' + max_depth=20 as models/rf_m4.pkl "
      f"({os.path.getsize('models/rf_m4.pkl')/1e6:.0f} MB)")
rf_ablation_df

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


default (max_features=1.0)            : fit=  37.1s  pickle=  4568.4MB  val_R2=0.8913  test_R2=0.8919  test_MAPE=11.67%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


max_features='sqrt'                   : fit=   9.2s  pickle=  4710.4MB  val_R2=0.8917  test_R2=0.8917  test_MAPE=11.66%


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


max_features='sqrt' + max_depth=20    : fit=   6.5s  pickle=  1546.2MB  val_R2=0.8892  test_R2=0.8896  test_MAPE=11.93%

shipping max_features='sqrt' + max_depth=20 as models/rf_m4.pkl (1546 MB)


,config,fit_seconds,pickle_mb,val_r2,test_r2,test_mae,test_mape,test_mdape
0,default (max_features=1.0),37.127619,4568.431139,0.891347,0.891853,159277.266248,0.116663,0.077062
1,max_features='sqrt',9.218823,4710.364723,0.891739,0.891666,160015.595878,0.116592,0.077426
2,max_features='sqrt' + max_depth=20,6.540720,1546.161427,0.889156,0.889592,163075.327148,0.119272,0.079352


Reading the three rows together: `max_features='sqrt'` is roughly a wash on accuracy here (test R2 0.8919 -> 0.8917, test MAPE 11.67% -> 11.66%, both essentially flat) but still **4.0x faster to fit** (37.1s -> 9.2s) -- decorrelating the trees didn't cost anything, and on this machine the speedup shows up even more starkly than the accuracy tradeoff does. What it did *not* fix is pickle size: 4568MB (default) vs. 4710MB (`sqrt`), essentially unchanged, if anything slightly larger. That's the same tell as before: pickle size was never really about feature width the way the notebook 6 writeup assumed (reasonably, at the time) -- it's dominated by how many *nodes* 200 unconstrained, full-depth trees grow across the training set, and `max_features` only controls how many columns are considered per split, not how deep the tree is allowed to go. Adding `max_depth=20` on top cuts the pickle to 1546MB (a 67% reduction versus the unconstrained `sqrt` version) and roughly halves fit time again (9.2s -> 6.5s), at a small, honest cost: test R2 drops to 0.8896 and MAPE rises to 11.93%. That's the version this notebook ships, since a multi-gigabyte model isn't a realistic artifact to hand off even when its three-more-significant-digit accuracy is marginally better. (One more thing worth naming honestly: this machine fits 200 trees on the full m4 training set in well under a minute in every configuration, versus 5-77 minutes in the sandbox environment where this notebook was first built -- a hardware/CPU-core difference between the two machines, not anything about the code.)

## 14. Full Model Lineup on m4

In [15]:
models = {}

lr_pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", LinearRegression())])
lr_pipeline.fit(X_train, y_train)
models["LinearRegression"] = lr_pipeline

dt_pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_pipeline.fit(X_train, y_train)
models["DecisionTree"] = dt_pipeline

models["RandomForest"] = rf_m4_pipeline  # from Section 13

xgb_param_grid = [
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": 6, "learning_rate": 0.03},
]
lgbm_param_grid = [
    {"n_estimators": 300, "max_depth": -1, "num_leaves": 31, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": -1, "num_leaves": 63, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": -1, "num_leaves": 127, "learning_rate": 0.03},
]

def tune_on_val(model_class, param_grid, label, **extra):
    best_params, best_val_r2, best_pipeline = None, -np.inf, None
    for params in param_grid:
        pipe = Pipeline([("preprocess", make_preprocessor()), ("model", model_class(random_state=RANDOM_STATE, **params, **extra))])
        pipe.fit(X_train, y_train)
        val_r2 = r2_score(y_val, pipe.predict(X_val))
        print(f"  {label} {params} -> val R2={val_r2:.4f}")
        if val_r2 > best_val_r2:
            best_params, best_val_r2, best_pipeline = params, val_r2, pipe
    print(f"  best {label}: {best_params} (val R2={best_val_r2:.4f})")
    return best_pipeline

xgb_pipeline = tune_on_val(XGBRegressor, xgb_param_grid, "XGBoost")
models["XGBoost"] = xgb_pipeline
joblib.dump(xgb_pipeline, "models/xgb_m4.pkl")

lgbm_pipeline = tune_on_val(LGBMRegressor, lgbm_param_grid, "LightGBM", verbosity=-1)
models["LightGBM"] = lgbm_pipeline
joblib.dump(lgbm_pipeline, "models/lgbm_m4.pkl")

overall_rows = []
predictions = {}
for name, pipe in models.items():
    preds = pipe.predict(X_test)
    predictions[name] = preds
    m = evaluate(pipe, X_test, y_test)
    overall_rows.append({"model": name, "band": "overall", "n_rows": len(y_test), **m})
    print(f"{name:>16s}: R2={m['r2']:.4f}  MAE=${m['mae']:,.0f}  MAPE={m['mape']:.2%}  MdAPE={m['mdape']:.2%}")

overall_df = pd.DataFrame(overall_rows)

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pa

  XGBoost {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1} -> val R2=0.8816


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  XGBoost {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05} -> val R2=0.8889


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


...

  LightGBM {'n_estimators': 300, 'max_depth': -1, 'num_leaves': 31, 'learning_rate': 0.1} -> val R2=0.8963


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  LightGBM {'n_estimators': 500, 'max_depth': -1, 'num_leaves': 63, 'learning_rate': 0.05} -> val R2=0.9006


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  LightGBM {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} -> val R2=0.9036
  best LightGBM: {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} (val R2=0.9036)
LinearRegression: R2=0.7183  MAE=$320,433  MAPE=29.60%  MdAPE=22.19%
    DecisionTree: R2=0.7973  MAE=$218,923  MAPE=16.13%  MdAPE=10.65%
    RandomForest: R2=0.8896  MAE=$163,075  MAPE=11.93%  MdAPE=7.94%
         XGBoost: R2=0.8958  MAE=$166,402  MAPE=12.50%  MdAPE=8.68%
        LightGBM: R2=0.9053  MAE=$156,418  MAPE=11.78%  MdAPE=8.15%


## 15. Headline Comparison: m3 (Before) vs. m4 (After the Overhaul)

The m3 numbers are the already-established, previously reported results from notebooks 5 and 6 (LightGBM R2=0.8933, Random Forest R2=0.8773, and so on), hardcoded here so the comparison is a straight read rather than a re-run of an unrelated feature set.

In [16]:
m3_baseline = {
    "LinearRegression": {"r2": 0.821782, "mae": 248655.75, "mape": 0.226937, "mdape": 0.160824},
    "DecisionTree":     {"r2": 0.774068, "mae": 231738.96, "mape": 0.169448, "mdape": 0.112363},
    "RandomForest":     {"r2": 0.877303, "mae": 169443.68, "mape": 0.122315, "mdape": 0.078733},
    "XGBoost":          {"r2": 0.863357, "mae": 203261.33, "mape": 0.162707, "mdape": 0.115994},
    "LightGBM":         {"r2": 0.893299, "mae": 166210.38, "mape": 0.124633, "mdape": 0.088176},
}

comparison_rows = []
for name in models:
    m3 = m3_baseline[name]
    m4 = overall_df.set_index("model").loc[name]
    comparison_rows.append({
        "model": name,
        "m3_r2": m3["r2"], "m4_r2": m4["r2"], "r2_delta": m4["r2"] - m3["r2"],
        "m3_mape": m3["mape"], "m4_mape": m4["mape"], "mape_delta_pp": (m4["mape"] - m3["mape"]) * 100,
        "m3_mae": m3["mae"], "m4_mae": m4["mae"], "mae_delta": m4["mae"] - m3["mae"],
    })
comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df = comparison_df.sort_values("m4_r2", ascending=False)
comparison_df

,m3_r2,m4_r2,r2_delta,m3_mape,m4_mape,mape_delta_pp,m3_mae,m4_mae,mae_delta
model,,,,,,,,,
LightGBM,0.893299,0.905303,0.012004,0.124633,0.117759,-0.687371,166210.38,156417.521677,-9792.858323
XGBoost,0.863357,0.895772,0.032415,0.162707,0.124986,-3.772069,203261.33,166401.909234,-36859.420766
RandomForest,0.877303,0.889592,0.012289,0.122315,0.119272,-0.304349,169443.68,163075.327148,-6368.352852
DecisionTree,0.774068,0.797306,0.023238,0.169448,0.161338,-0.811046,231738.96,218923.136682,-12815.823318
LinearRegression,0.821782,0.718250,-0.103532,0.226937,0.296031,6.909430,248655.75,320433.296905,71777.546905


Four of five models improved, some substantially: XGBoost's R2 climbed from 0.8634 to 0.8958 (+0.032) and its MAPE dropped 3.77 points (16.27% -> 12.50%), the largest single-model gain in the lineup. LightGBM, already the best model on m3, improved further (R2 0.8933 -> 0.9053, MAPE 12.46% -> 11.78%) and remains the top model on m4. Random Forest (R2 0.8773 -> 0.8896, MAPE 12.23% -> 11.93%) and Decision Tree (R2 0.7741 -> 0.7973, MAPE 16.94% -> 16.13%) both improved modestly.

**Linear Regression is the one model that got worse, and not by a small margin** (R2 0.8218 -> 0.7183, a 0.10 drop; MAPE +6.91 points, 22.69% -> 29.60%; MAE +$71,777). Target encoding (Fix #6) is still the most likely primary driver -- five target-encoded location columns that are meaningfully collinear with each other expose a linear model much more than a tree that can just pick whichever one splits best at each node. But the size of this drop, noticeably larger than first observed in an earlier run of this same notebook, points to a second contributing factor: the winning training window here is 24 months (Section 11) versus a shorter window elsewhere, and this feature set still has no temporal/seasonality feature at all (see the wk10 domain-features discussion) -- a linear model has no way to separate a genuine price effect from a market-regime shift across two extra years of data, while tree-based models can partly compensate by splitting on other correlated signals. This is a real, measured tradeoff, not an oversight: every tree-based model, and specifically the two candidates worth deploying, got better, and Linear Regression was never a deployment candidate to begin with, it exists in this lineup as the interpretable baseline the task prompt calls for.

## 16. Evaluation on the Team's Prescribed Price Bands

Per team decision (for consistency across everyone's notebooks going forward): **under \$500K, \$500K-\$1M, \$1M-\$2M, \$2M+**, replacing the quantile-based Q1-Q5 bands used in notebook 6. Fixed dollar bands aren't balanced in row count the way quantile bands are (the middle two bands dominate this test set, `$500K-$1M` and `$1M-$2M` together are 64% of the rows), but they're stable across notebooks and re-runs, which quantile bands built from that run's own test set are not, and they map directly to how stakeholders actually talk about price tiers.

In [17]:
TEAM_PRICE_BAND_EDGES = [0, 500_000, 1_000_000, 2_000_000, np.inf]
TEAM_PRICE_BAND_LABELS = ["under $500K", "$500K-$1M", "$1M-$2M", "$2M+"]
team_bands = pd.cut(y_test, bins=TEAM_PRICE_BAND_EDGES, labels=TEAM_PRICE_BAND_LABELS)

band_rows = []
for name, preds in predictions.items():
    for label in TEAM_PRICE_BAND_LABELS:
        mask = (team_bands == label).values
        if mask.sum() == 0:
            continue
        y_band, preds_band = y_test[mask], preds[mask]
        band_rows.append({
            "model": name, "band": label, "n_rows": int(mask.sum()),
            "r2": np.nan, "mae": mean_absolute_error(y_band, preds_band),
            "mape": mean_absolute_percentage_error(y_band, preds_band),
            "mdape": float(np.median(np.abs((y_band - preds_band) / y_band))),
        })
band_df = pd.DataFrame(band_rows)
print("row counts per band:")
print(band_df[band_df["model"] == "LightGBM"][["band", "n_rows"]].to_string(index=False))
print("\nMAPE by model x band:")
band_df.pivot(index="model", columns="band", values="mape")[TEAM_PRICE_BAND_LABELS].round(4)

row counts per band:
       band  n_rows
under $500K    1629
  $500K-$1M    4894
    $1M-$2M    3745
       $2M+    1608

MAPE by model x band:


band,under $500K,$500K-$1M,$1M-$2M,$2M+
model,,,,
DecisionTree,0.1923,0.1319,0.1654,0.2101
LightGBM,0.1488,0.0977,0.1185,0.1456
LinearRegression,0.5294,0.3049,0.2111,0.2302
RandomForest,0.1462,0.0965,0.1218,0.1554
XGBoost,0.1551,0.1043,0.1259,0.1552


In [18]:
wk9_metrics_summary = pd.concat([overall_df, band_df], ignore_index=True)
wk9_metrics_summary = wk9_metrics_summary[["model", "band", "n_rows", "r2", "mae", "mape", "mdape"]]
wk9_metrics_summary.to_csv("Deliverables/wk9_m4_metrics_summary.csv", index=False)
print(f"saved Deliverables/wk9_m4_metrics_summary.csv ({len(wk9_metrics_summary)} rows)")

saved Deliverables/wk9_m4_metrics_summary.csv (25 rows)


Same pattern as notebook 6's quantile-band breakdown, now on bands the team will actually keep using, though the specific ranking is more nuanced than a single earlier run suggested. Random Forest has the edge at the low end (`under $500K`: 14.62% vs. LightGBM's 14.88%; `$500K-$1M`: 9.65% vs. 9.77%, both close), while LightGBM pulls ahead at the high end (`$1M-$2M`: 11.85% vs. RF's 12.18%; `$2M+`: 14.56% vs. RF's 15.54%). Critically, **the worst band is not the same for every model** here -- LightGBM and Linear Regression are hardest hit at `under $500K` (14.88% and 52.94% respectively), but Decision Tree, Random Forest, and XGBoost are all hardest hit at `$2M+` instead (21.01%, 15.54%, and 15.52%). That's a genuinely different finding from the single-cutoff read this notebook first shipped with, which claimed the cheapest band was universally worst -- worth remembering as a caution against generalizing too far from one run: the *direction* of the story (entry-level and luxury homes are both harder than the middle of the market) held up, but the specific 'which end is worse, and for which model' claim did not, and the band-by-model table is the source of truth going forward, not the prose summary of it.

## Reflection

**A note on these numbers.** This notebook was first built and validated in a separate sandbox environment, then re-run end-to-end on my own machine. The two runs produced consistently different -- but each internally consistent and leak-free -- results, which turned out to trace back to a genuine data snapshot mismatch: my local `CRMLSData/` folder has one more month of sales (`CRMLSSold202605.csv`, May 2026) than the folder used for the original validation run. That single extra month shifts the chronological test/val/train boundaries for every downstream step, which is enough on its own to move every metric in this notebook. The numbers below are from my own machine, on my own current data snapshot, and are the authoritative numbers for this notebook going forward. This is exactly the reproducibility risk the AVM best-practices doc flags (pin the environment, note the data snapshot) -- addressing that properly is a next step I'm deliberately deferring for now rather than solving inside this notebook.

**What actually moved the needle.** Two fixes account for nearly all of the measured improvement. The school district de-duplication (Fix #4) is still the one to lead with in any conversation about this notebook: it's not a tuning tweak, it's a correction to a genuine defect that silently duplicated 40% of the m2 test set, and while its effect on m3's headline numbers can't be measured directly (m3 didn't use the school district feature), it's the fix most likely to have been quietly distorting any future notebook built on m2 instead of m3. Target encoding (Fix #6) has the largest directly measured effect: XGBoost's R2 up 0.032 and MAPE down 3.77 points is the biggest single-model swing in the notebook, and it came from a preprocessing change, not a modeling one.

**What was correctly suspected but only partially fixable.** The Random Forest ablation confirms the `max_features=1.0` default was a real problem, fixed essentially for free (4x faster fit, flat accuracy). It also confirms, again, that pickle size was never primarily about `max_features` -- it's unconstrained tree depth, and fixing that required a second fix (`max_depth=20`) with a small, honest accuracy cost.

**What didn't need fixing.** `AssociationFee` imputation and lot-size recovery were checked directly and found not to be real problems -- reported as closed questions, not silently fixed or silently skipped.

**The one fix that made something worse, on purpose, with the tradeoff shown -- and it's bigger than first thought.** Linear Regression's R2 dropped from 0.8218 to 0.7183 under target encoding, a much larger decline than an earlier run of this same pipeline showed (which saw a ~0.03 drop). The larger training window this run's data supports (24 months, versus 12-18 assumed or found elsewhere) is the most likely reason the gap widened: more history means more market-regime drift that a linear model with no temporal feature at all can't separate from a real price effect, while tree models partially compensate. That's a concrete argument for the cyclical month/seasonality feature flagged as a wk10 next step, not just a nice-to-have. Reported here plainly because Linear Regression isn't the deployment candidate, and every tree-based model got better.

**Bottom line.** LightGBM on m4 (R2=0.9053, MAPE=11.78%, MdAPE=8.15%) is the best model produced across every version of this pipeline so far, ahead of LightGBM on m3 (R2=0.8933) and every other m4 alternative including Random Forest (R2=0.8896, MAPE=11.93%). The most valuable output of this notebook is still arguably Section 9 (the school-district join fix, caught by going back to raw data instead of trusting a cached checkpoint) -- but a close second, discovered only during this re-run, is the reminder that a result computed once, on one machine, on one data pull, is an anecdote until it's been reproduced. Pinning the environment and recording the exact data snapshot used (both flagged as wk10/deployment follow-ups) would have caught this discrepancy immediately instead of requiring a debugging conversation.